In [0]:

# GOLD 6.3 — Weekly Sector Performance
# 
# Non-overlapping 5-trading-day blocks → compound return per sector.
# KPIs: spot-check accuracy, winner/loser identification, sector alpha.


from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark.sql("USE CATALOG iran_israel_capstone_project")
silver = spark.table("silver.daily_market_clean")
events = spark.table("silver.event_dim")

# Sectors to track (close column → friendly name)
SECTORS = {
    "nifty_close":       "Nifty",
    "niftybank_close":   "NiftyBank",
    "niftyenergy_close": "NiftyEnergy",
    "niftyauto_close":   "NiftyAuto",
    "niftyit_close":     "NiftyIT",
    "hal_close":         "HAL",
    "indigo_close":      "IndiGo",
    "asianpaint_close":  "AsianPaint",
    "ongc_close":        "ONGC",
}

# ============================================================
# 1. ASSIGN 5-DAY WEEK BLOCKS
# ============================================================
date_win = Window.orderBy("trade_date")

daily = (
    silver
    .select(
        "trade_date",
        *[F.col(c) for c in SECTORS.keys()],
        "event_id", "severity",
    )
    .withColumn("row_num", F.row_number().over(date_win))
    .withColumn("week_num", F.ceil(F.col("row_num") / 5))
)

# Week boundaries: start/end dates and day count
week_bounds = (
    daily
    .groupBy("week_num")
    .agg(
        F.min("trade_date").alias("week_start"),
        F.max("trade_date").alias("week_end"),
        F.count("*").alias("days_in_week"),
    )
)

# ============================================================
# 2. WEEKLY COMPOUND RETURNS
# ============================================================
# Get the LAST close of each week for every sector
row_in_week = Window.partitionBy("week_num").orderBy(F.desc("trade_date"))

last_day = (
    daily
    .withColumn("rn", F.row_number().over(row_in_week))
    .filter(F.col("rn") == 1)
    .drop("rn")
)

# Lag: previous week's last close for each sector
week_win = Window.orderBy("week_num")

weekly = last_day
for col_name, friendly in SECTORS.items():
    prev_col = f"prev_{col_name}"
    ret_col  = f"{friendly}_weekly_return"
    weekly = (
        weekly
        .withColumn(prev_col, F.lag(col_name).over(week_win))
        .withColumn(
            ret_col,
            F.round((F.col(col_name) / F.col(prev_col) - 1) * 100, 4)
        )
        .drop(prev_col)
    )

# Join week boundaries
weekly = weekly.join(week_bounds, on="week_num", how="inner")

# ============================================================
# 3. FLAG HIGH/CRITICAL EVENT WEEKS
# ============================================================
hc_events = (
    events
    .filter(F.col("severity").isin("HIGH", "CRITICAL"))
    .select("event_date", "severity")
)

# A week is a conflict week if ANY HIGH/CRITICAL event falls within it
conflict_weeks = (
    weekly.alias("w")
    .crossJoin(hc_events.alias("e"))
    .filter(
        (F.col("e.event_date") >= F.col("w.week_start"))
        & (F.col("e.event_date") <= F.col("w.week_end"))
    )
    .select(F.col("w.week_num"))
    .distinct()
)

weekly = (
    weekly
    .join(conflict_weeks, on="week_num", how="left_semi")
    .withColumn("is_conflict_week", F.lit(True))
).unionByName(
    weekly
    .join(conflict_weeks, on="week_num", how="left_anti")
    .withColumn("is_conflict_week", F.lit(False))
)

# ============================================================
# 4. UNPIVOT TO LONG FORMAT + SECTOR ALPHA
# ============================================================
# Non-benchmark sectors for ranking
NON_BENCH = {k: v for k, v in SECTORS.items() if v != "Nifty"}

ret_cols = [f"{v}_weekly_return" for v in NON_BENCH.values()]
stack_expr = ", ".join([f"'{v}', `{v}_weekly_return`" for v in NON_BENCH.values()])
n = len(NON_BENCH)

sector_long = (
    weekly
    .select(
        "week_num", "week_start", "week_end", "is_conflict_week",
        "Nifty_weekly_return",
        *ret_cols,
    )
    .selectExpr(
        "week_num", "week_start", "week_end", "is_conflict_week",
        "Nifty_weekly_return",
        f"stack({n}, {stack_expr}) as (sector, sector_weekly_return)",
    )
    .filter(F.col("sector_weekly_return").isNotNull())
    .withColumn(
        "sector_alpha",
        F.round(F.col("sector_weekly_return") - F.col("Nifty_weekly_return"), 4)
    )
)

# ============================================================
# 5. WINNER / LOSER IDENTIFICATION
# ============================================================
rank_win = Window.partitionBy("week_num").orderBy(F.desc("sector_weekly_return"))
rank_asc = Window.partitionBy("week_num").orderBy("sector_weekly_return")

sector_ranked = (
    sector_long
    .filter(F.col("is_conflict_week") == True)
    .withColumn("rank_top", F.row_number().over(rank_win))
    .withColumn("rank_bot", F.row_number().over(rank_asc))
    .withColumn(
        "position",
        F.when(F.col("rank_top") <= 2, "TOP_2")
         .when(F.col("rank_bot") <= 2, "BOTTOM_2")
         .otherwise("MIDDLE")
    )
)

# KPI: defence (HAL) + energy (NiftyEnergy/ONGC) in top-2 for >= 60% of conflict weeks
defence_energy = ["HAL", "NiftyEnergy", "ONGC"]
conflict_week_count = sector_ranked.select("week_num").distinct().count()

top2_weeks_with_de = (
    sector_ranked
    .filter((F.col("rank_top") <= 2) & F.col("sector").isin(defence_energy))
    .select("week_num")
    .distinct()
    .count()
)

de_pct = top2_weeks_with_de / conflict_week_count * 100 if conflict_week_count > 0 else 0
de_pass = de_pct >= 60

print("\n" + "="*65)
print("  WINNER/LOSER IDENTIFICATION (HIGH/CRITICAL event weeks)")
print("="*65)
print(f"  Conflict weeks analysed          : {conflict_week_count}")
print(f"  Weeks w/ Defence/Energy in top-2  : {top2_weeks_with_de}")
print(f"  Rate                              : {de_pct:.1f}%")
print(f"  Target                            : >= 60%")
print(f"  Status                            : {'✅' if de_pass else '❌'}")
print("="*65)

# Show top-2 and bottom-2 per conflict week
print("\n── Top-2 / Bottom-2 per Conflict Week ──")
(
    sector_ranked
    .filter(F.col("position") != "MIDDLE")
    .select("week_start", "week_end", "sector", "sector_weekly_return", "sector_alpha", "position")
    .orderBy("week_start", "position", F.desc("sector_weekly_return"))
    .show(100, truncate=False)
)

# ============================================================
# 6. PERSIST
# ============================================================
spark.sql("CREATE SCHEMA IF NOT EXISTS iran_israel_capstone_project.gold")

(
    sector_long.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("iran_israel_capstone_project.gold.gold_sector_performance")
)

row_count = sector_long.count()
print(f"\n✅ gold.gold_sector_performance written ({row_count} rows)")
display(sector_long.filter(F.col("is_conflict_week") == True).orderBy("week_start", "sector"))

In [0]:
# ============================================================
# SPOT-CHECK — HAL & IndiGo Weekly Return
# ============================================================
# Pick first conflict week, manually verify compound return
# matches to 2 decimal places.
# ============================================================

# Get the first conflict week's boundaries
first_cw = (
    sector_long
    .filter(F.col("is_conflict_week") == True)
    .select("week_num", "week_start", "week_end")
    .distinct()
    .orderBy("week_num")
    .first()
)
wk_num   = first_cw["week_num"]
wk_start = first_cw["week_start"]
wk_end   = first_cw["week_end"]

print(f"Spot-check week: {wk_num} ({wk_start} → {wk_end})")

# Get the previous week's last close (baseline for compound return)
prev_week_last_day = (
    daily
    .filter(F.col("week_num") == wk_num - 1)
    .orderBy(F.desc("trade_date"))
    .first()
)

# Get all days in this week
week_days = (
    daily
    .filter(F.col("week_num") == wk_num)
    .orderBy("trade_date")
    .select("trade_date", "hal_close", "indigo_close")
    .collect()
)

for ticker, col_name in [("HAL", "hal_close"), ("IndiGo", "indigo_close")]:
    baseline = prev_week_last_day[col_name]
    print(f"\n{'='*60}")
    print(f"  {ticker} Spot-Check")
    print(f"{'='*60}")
    print(f"  Baseline (prev week close): {baseline:.2f}")
    print(f"  Daily closes:")
    for row in week_days:
        daily_ret = (row[col_name] / baseline - 1) * 100
        print(f"    {row['trade_date']}  close={row[col_name]:.2f}  cum_return={daily_ret:+.4f}%")
        baseline_for_display = row[col_name]

    # Final compound return
    end_close = week_days[-1][col_name]
    start_close = prev_week_last_day[col_name]
    manual_return = round((end_close / start_close - 1) * 100, 4)

    # Get computed value
    computed = (
        sector_long
        .filter((F.col("week_num") == wk_num) & (F.col("sector") == ticker))
        .select("sector_weekly_return")
        .first()
    )["sector_weekly_return"]

    match = abs(manual_return - computed) < 0.01
    print(f"\n  Manual  return: {manual_return:+.4f}%")
    print(f"  Computed return: {computed:+.4f}%")
    print(f"  Match (2 dp)  : {'' if match else ''}")

print("\n" + "="*60)
print("✅ Spot-check complete")